In [74]:
## Import Required Libraries
import pandas as pd
import numpy as np 
import ast 
import pickle

In [75]:
# Load the movie metadata and credits datasets.
movies = pd.read_csv("tmdb_5000_movies.csv") 
credits = pd.read_csv("tmdb_5000_credits.csv") 

In [76]:
movies.head(1) 

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [77]:
credits.head(1) 

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [78]:
# Merge the movies and credits datasets using the common movie title.
movies = movies.merge(credits, on="title") 

In [79]:
movies.head(2) 

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [80]:
# Retain only the features required for the recommendation system.
movies = movies[["title", "movie_id", "genres", "overview", "keywords", "cast", "crew", "popularity"]] 

In [81]:
# Check for missing values in the selected features.
movies.isnull().sum() 

title         0
movie_id      0
genres        0
overview      3
keywords      0
cast          0
crew          0
popularity    0
dtype: int64

In [82]:
# Remove records containing missing values.
movies.dropna(inplace = True) 

In [83]:
# Check for duplicate records in the dataset.
movies.duplicated().sum() 

np.int64(0)

## Extract Top Popular Movies

The recommendation system includes a **Popular Movies** section on the home page. To prepare this dataset, the movies are ranked using the TMDB popularity score, and the top 20 movies are selected. Additional metadata, such as poster URLs, is then retrieved using the TMDB API.

In [84]:
# Select the top 20 most popular movies based on the TMDB popularity score.
popular_movies = movies.sort_values(by="popularity", ascending=False).head(20)[["movie_id", "title"]].reset_index(drop=True) 

### Fetch Movie Posters

The original dataset does not include poster image URLs. Therefore, the TMDB API is used to retrieve poster URLs using each movie's unique TMDB movie ID. These URLs will later be displayed in the Flask application.

In [85]:
# Import the libraries required for API requests and environment variable management.
from dotenv import load_dotenv
import os
import requests

In [86]:
# Load the TMDB API key from the environment file.
load_dotenv()

# Retrieve the poster URL for a given TMDB movie ID.
API_KEY = os.getenv("TMDB_API_KEY")

def fetch_poster(movie_id):

    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={API_KEY}"

    data = requests.get(url).json()

    if data["poster_path"]:

        return "https://image.tmdb.org/t/p/w500" + data["poster_path"]

    return None 

In [87]:
# Fetch poster URLs for the selected popular movies.
popular_movies["poster"] = popular_movies["movie_id"].apply(fetch_poster)

In [88]:
# Preview the final popular movies dataset.
popular_movies

,movie_id,title,poster
0,211672,Minions,https://image.tmdb.org/t/p/w500/dr02BdCNAUPVU0...
1,157336,Interstellar,https://image.tmdb.org/t/p/w500/yQvGrMoipbRodd...
2,293660,Deadpool,https://image.tmdb.org/t/p/w500/3E53WEZJqP6aM8...
3,118340,Guardians of the Galaxy,https://image.tmdb.org/t/p/w500/r7vmZjiyZw9rpJ...
4,76341,Mad Max: Fury Road,https://image.tmdb.org/t/p/w500/hA2ple9q4qnwxp...
5,135397,Jurassic World,https://image.tmdb.org/t/p/w500/rhr4y79GpxQF9I...
6,22,Pirates of the Caribbean: The Curse of the Bla...,https://image.tmdb.org/t/p/w500/poHwCZeWzJCShH...
7,119450,Dawn of the Planet of the Apes,https://image.tmdb.org/t/p/w500/mSmAc9G25fhOHH...
8,131631,The Hunger Games: Mockingjay - Part 1,https://image.tmdb.org/t/p/w500/4FAA18ZIja70d1...
9,177572,Big Hero 6,https://image.tmdb.org/t/p/w500/2mxS4wUimwlLmI...


## Text Preprocessing

The selected features are cleaned and transformed into a structured format that can be used for feature engineering. This includes extracting relevant information from nested JSON-like columns and preparing textual features for recommendation generation.

In [89]:
# Select the features required for building the recommendation system.
movies = movies[["title", "movie_id", "genres", "overview", "keywords", "cast", "crew"]] 

In [90]:
# Preview the selected features.
movies.head() 

,title,movie_id,genres,overview,keywords,cast,crew
0,Avatar,19995,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","In the 22nd century, a paraplegic Marine is di...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,Pirates of the Caribbean: At World's End,285,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","Captain Barbossa, long believed to be dead, ha...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,Spectre,206647,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",A cryptic message from Bond’s past sends him o...,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,The Dark Knight Rises,49026,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",Following the death of District Attorney Harve...,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,John Carter,49529,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","John Carter is a war-weary, former military ca...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


### Extract Genre and Keyword Information

The `genres` and `keywords` columns contain JSON-like strings with additional metadata. Since the recommendation system only requires the names of the genres and keywords, the remaining information is discarded by extracting only the `name` field from each entry.

In [91]:
# Extract only the genre names from the JSON-like structure.
def convert(obj): 
    L = [] 
    for el in ast.literal_eval(obj): 
        L.append(el["name"]) 
    return L 

In [92]:
# Apply the transformation to the genres column.
movies["genres"] = movies["genres"].apply(convert) 

In [93]:
# Apply the transformation to the keywords column.
movies["keywords"] = movies["keywords"].apply(convert) 

### Extract Cast and Director Information

The `cast` and `crew` columns contain detailed information about the people involved in a movie. To improve recommendation quality while reducing noise, only the names of the first three cast members and the movie director are extracted, as they have the greatest influence on a movie's identity.

In [94]:
# Extract the names of the first three cast members.
def extract_top_cast(obj):
    L = [] 
    counter = 0 
    for el in ast.literal_eval(obj): 
        L.append(el["name"]) 
        counter += 1 
        if counter == 3: 
            break 
    return L 

# Apply the transformation to the cast column.
movies["cast"] = movies["cast"].apply(extract_top_cast) 

In [95]:
# Extract the director's name from the crew information.
def extract_director(obj):
    L = [] 
    for el in ast.literal_eval(obj): 
        if el["job"] == "Director":
            L.append(el["name"]) 
            break
    return L 

# Apply the transformation to the crew column.
movies["crew"] = movies["crew"].apply(extract_director) 

## Tokenize Movie Overview

The movie overview is split into individual words to prepare the textual data for feature engineering. This tokenized representation will later be combined with other movie attributes to create the final text representation used by the recommendation system.

In [96]:
# Split the overview text into individual words.
movies["overview"] = movies["overview"].apply(lambda x: x.split()) 

In [97]:
movies.head() 

,title,movie_id,genres,overview,keywords,cast,crew
0,Avatar,19995,"[Action, Adventure, Fantasy, Science Fiction]","[In, the, 22nd, century,, a, paraplegic, Marin...","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,Pirates of the Caribbean: At World's End,285,"[Adventure, Fantasy, Action]","[Captain, Barbossa,, long, believed, to, be, d...","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,Spectre,206647,"[Action, Adventure, Crime]","[A, cryptic, message, from, Bond’s, past, send...","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,The Dark Knight Rises,49026,"[Action, Crime, Drama, Thriller]","[Following, the, death, of, District, Attorney...","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,John Carter,49529,"[Action, Adventure, Science Fiction]","[John, Carter, is, a, war-weary,, former, mili...","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


### Create Movie Tags

The extracted features are cleaned and combined into a single **tags** column that represents each movie. This consolidated text serves as the input for the vectorization process used by the recommendation system.

In [98]:
# Remove spaces from multi-word names to preserve them as single tokens.
def no_space(obj):
    L = []
    for name in obj:
        L.append(name.replace(" ", ""))
    return L 

In [99]:
# Apply the transformation to all extracted textual features.
movies["genres"] = movies["genres"].apply(no_space) 
movies["keywords"] = movies["keywords"].apply(no_space) 
movies["cast"] = movies["cast"].apply(no_space) 
movies["crew"] = movies["crew"].apply(no_space) 

In [100]:
# Combine all textual features into a single column for recommendation generation.
movies["tags"] = movies["overview"] + movies["genres"] + movies["keywords"] + movies["cast"] + movies["crew"] 

In [101]:
# Create the final dataset containing the movie title, ID, and combined text representation.
new_df = movies[["title", "movie_id", "tags"]]

In [102]:
# Preview the final dataset.
new_df.head() 

,title,movie_id,tags
0,Avatar,19995,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,285,"[Captain, Barbossa,, long, believed, to, be, d..."
2,Spectre,206647,"[A, cryptic, message, from, Bond’s, past, send..."
3,The Dark Knight Rises,49026,"[Following, the, death, of, District, Attorney..."
4,John Carter,49529,"[John, Carter, is, a, war-weary,, former, mili..."


In [103]:
# Convert the list of tokens into a single text string.
new_df["tags"] = new_df["tags"].apply(lambda x: " ".join(x)) 

In [104]:
# Convert the text to lowercase for consistency.
new_df["tags"] = new_df["tags"].apply(lambda x: x.lower()) 

In [105]:
# Preview the final dataset.
new_df.head() 

,title,movie_id,tags
0,Avatar,19995,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,285,"captain barbossa, long believed to be dead, ha..."
2,Spectre,206647,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,49026,following the death of district attorney harve...
4,John Carter,49529,"john carter is a war-weary, former military ca..."


In [106]:
# Save the processed datasets for the recommendation system.
pickle.dump(popular_movies, open("popular_movies.pkl", "wb")) 
pickle.dump(new_df, open("new_df.pkl", "wb")) 

## Conclusion

The movie metadata has been cleaned, transformed, and combined into a unified text representation (`tags`) for each movie. Additionally, the top popular movies dataset has been prepared with poster URLs fetched from the TMDB API. These processed datasets will serve as the input for the vectorization and recommendation pipeline in the subsequent notebook.